<a href="https://colab.research.google.com/github/Bia1210/previsao-churn-streaming/blob/main/analisar_churn_clientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Etapa 1: ler o dicionário

In [30]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
dic = pd.read_csv("/content/dicionario_de_dados.csv")
display(dic)

,coluna,tipo,descricao
0,id_cliente,texto,Identificador sintético do cliente (sem dados pessoais)
1,data_cadastro,data,Data de início da assinatura
2,tempo_assinatura_meses,numérica,Meses de assinatura até a data de corte (01/01/2026)
3,idade,numérica,Idade em anos (1% nulos)
4,faixa_renda,categórica,Faixa de renda declarada (3% nulos)
5,uf,categórica,Unidade da federação
6,plano,categórica,"Básico, Padrão, Premium ou Família"
7,valor_mensal,numérica,Preço mensal do plano em R$
8,ciclo_faturamento,categórica,Mensal ou Anual
9,metodo_pagamento,categórica,Forma de pagamento principal


Etapa 3: auditoria de vazamento (base de 450 linhas)



In [31]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier

# Carregando o dataset de 50 mil clientes
d = pd.read_csv("/content/dataset_streaming_churn_50k.csv")

# Amostrar 450 linhas para manter a consistência da auditoria rápida
d_sample = d.sample(n=450, random_state=42).copy()

# Alvo já está mapeado em 'cancelou_60d'
target = "cancelou_60d"

# Mapeamento com as colunas reais do dataset
cat = ["plano", "ciclo_faturamento", "metodo_pagamento", "origem_cadastro",
       "dispositivo_principal", "faixa_renda", "uf"]
num = ["idade", "tempo_assinatura_meses", "qtd_perfis", "score_engajamento"]

def avalia(cat, num, rotulo):
    # Pipelines de pré-processamento com tratamento de dados ausentes (Imputer)
    cat_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    num_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    pre = ColumnTransformer([
        ("c", cat_transformer, cat),
        ("n", num_transformer, num)
    ])

    gb = GradientBoostingClassifier(max_depth=2, n_estimators=100, learning_rate=0.05,
                                    subsample=0.8, random_state=0)
    pipe = Pipeline([("p", pre), ("m", gb)])
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    s = cross_val_score(pipe, d_sample[cat + num], d_sample[target], cv=cv, scoring="average_precision")
    print(f"{rotulo}: PR-AUC {s.mean():.3f} ± {s.std():.3f}  (acaso = {d_sample[target].mean():.2f})")

avalia(cat, num, "SEM vazamento")
avalia(cat, num + ["dias_ativos_30d", "horas_assistidas_30d"], "COM vazamento")

SEM vazamento: PR-AUC 0.425 ± 0.083  (acaso = 0.15)
COM vazamento: PR-AUC 0.414 ± 0.079  (acaso = 0.15)


Resultado esperado: cerca de 0,25 sem vazamento e 0,97 com. A diferença entre os dois é a lição da etapa.


In [32]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
import pandas as pd

# 1. Definindo X e y usando o dataset de 50 mil clientes carregado em 'd'
target = "cancelou_60d"
X = d.drop(columns=[target, "id_cliente", "data_cadastro"])  # Remove identificadores irrelevantes para o modelo
y = d[target]

# 2. Dividindo em treino e teste
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Seu código a partir daqui com as variáveis agora devidamente definidas:
sem_eng = ["score_engajamento", "variacao_engajamento_30d"]
Xtr2, Xte2 = X_tr.drop(columns=sem_eng), X_te.drop(columns=sem_eng)

cat2 = Xtr2.select_dtypes("object").columns.tolist()
num2 = [c for c in Xtr2.columns if c not in cat2]

pre2 = ColumnTransformer([
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="ND")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat2),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num2)])

m2 = Pipeline([("pre", pre2), ("m", LogisticRegression(C=0.5, max_iter=2000))]).fit(Xtr2, y_tr)
p2 = m2.predict_proba(Xte2)[:, 1]
print("PR-AUC sem engajamento: %.3f  (com engajamento: 0.419)" % average_precision_score(y_te, p2))

PR-AUC sem engajamento: 0.362  (com engajamento: 0.419)


Versão que compara os dois modelos na mesma divisão

Rode depois das células do notebook principal, que já definem X_tr, X_te, y_tr, y_te, pre e os imports:

In [33]:
def pr_auc(Xa, Xb, nome):
    cat_ = Xa.select_dtypes("object").columns.tolist()
    num_ = [c for c in Xa.columns if c not in cat_]
    pre_ = ColumnTransformer([
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="ND")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_)])
    m = Pipeline([("pre", pre_), ("m", LogisticRegression(C=0.5, max_iter=2000))]).fit(Xa, y_tr)
    s = average_precision_score(y_te, m.predict_proba(Xb)[:, 1])
    print(f"{nome}: PR-AUC {s:.3f}")
    return s

sem_eng = ["score_engajamento", "variacao_engajamento_30d"]
com = pr_auc(X_tr, X_te, "COM engajamento")
sem = pr_auc(X_tr.drop(columns=sem_eng), X_te.drop(columns=sem_eng), "SEM engajamento")
print(f"Queda: {com - sem:.3f} ({(com - sem) / com:.0%} da PR-AUC)")

COM engajamento: PR-AUC 0.405
SEM engajamento: PR-AUC 0.362
Queda: 0.043 (11% da PR-AUC)


Desafio 3: vazamento de propósito

Rode depois da comparação acima, na mesma sessão (ele reutiliza a função pr_auc):

In [34]:
Xtr_v, Xte_v = X_tr.copy(), X_te.copy()
Xtr_v["vazamento"] = y_tr.values + np.random.default_rng(0).normal(0, 0.1, len(y_tr))
Xte_v["vazamento"] = y_te.values + np.random.default_rng(1).normal(0, 0.1, len(y_te))

pr_auc(Xtr_v, Xte_v, "COM VAZAMENTO")

COM VAZAMENTO: PR-AUC 1.000


np.float64(1.0)

Desafio 4: class_weight="balanced"

Rode na mesma sessão, depois das células do notebook:

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score

cat = X_tr.select_dtypes("object").columns.tolist()
num = [c for c in X_tr.columns if c not in cat]
pre = ColumnTransformer([
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="ND")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num)])

def avalia(cw, nome):
    m = Pipeline([("pre", pre), ("m", LogisticRegression(C=0.5, max_iter=2000, class_weight=cw))]).fit(X_tr, y_tr)
    p = m.predict_proba(X_te)[:, 1]
    pred = (p >= 0.5).astype(int)
    print(f"{nome}: PR-AUC {average_precision_score(y_te, p):.3f} | "
          f"precision {precision_score(y_te, pred):.3f} | recall {recall_score(y_te, pred):.3f} | "
          f"% sinalizados {pred.mean():.1%}")

avalia(None, "Sem class_weight")
avalia("balanced", "class_weight=balanced")

Sem class_weight: PR-AUC 0.405 | precision 0.587 | recall 0.150 | % sinalizados 3.6%
class_weight=balanced: PR-AUC 0.404 | precision 0.297 | recall 0.711 | % sinalizados 33.8%


Desafio 5: achar o corte que maximiza o lucro

In [36]:
CUSTO, VALOR, SUCESSO = 15, 120, 0.30

modelo_d5 = Pipeline([("pre", pre), ("m", LogisticRegression(C=0.5, max_iter=2000))]).fit(X_tr, y_tr)
p = modelo_d5.predict_proba(X_te)[:, 1]
y_arr = y_te.values

def lucro(corte):
    sel = p >= corte
    return (y_arr[sel].sum() * SUCESSO * VALOR) - (sel.sum() * CUSTO)

cortes = np.arange(0.05, 0.95, 0.01)
lucros = [lucro(c) for c in cortes]
melhor = cortes[int(np.argmax(lucros))]

print(f"Corte 0.50: lucro R$ {lucro(0.5):,.0f}")
print(f"Melhor corte: {melhor:.2f} -> lucro R$ {max(lucros):,.0f}, sinaliza {(p >= melhor).mean():.1%}")
print(f"Ponto de equilíbrio teórico: {CUSTO/(SUCESSO*VALOR):.3f}")
corte_teorico = CUSTO / (SUCESSO * VALOR)
print(f"Corte teórico {corte_teorico:.3f} -> lucro R$ {lucro(corte_teorico):,.0f}")

Corte 0.50: lucro R$ 3,312
Melhor corte: 0.43 -> lucro R$ 3,915, sinaliza 5.9%
Ponto de equilíbrio teórico: 0.417
Corte teórico 0.417 -> lucro R$ 3,732


Opção 2: escolher o corte em outra parte dos dados

Separamos um pedaço do treino só para escolher o corte. Essa parte se chama validação e funciona como um simulado. O teste real só entra no final, uma vez.

In [37]:
from sklearn.model_selection import train_test_split

In [38]:
# 1. Separa uma parte do treino para o "simulado"
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=42)


In [39]:
# 2. Treina só com a parte "fit" e prevê no simulado
m_val = Pipeline([("pre", pre), ("m", LogisticRegression(C=0.5, max_iter=2000))]).fit(X_fit, y_fit)
p_val = m_val.predict_proba(X_val)[:, 1]
yv = y_val.values

def lucro_val(corte):
    sel = p_val >= corte
    return (yv[sel].sum() * SUCESSO * VALOR) - (sel.sum() * CUSTO)


In [40]:
# 3. Escolhe o melhor corte no simulado
cortes = np.arange(0.05, 0.95, 0.01)
corte_val = cortes[int(np.argmax([lucro_val(c) for c in cortes]))]


In [41]:
# 4. Usa esse corte no teste, uma única vez
p_te = m_val.predict_proba(X_te)[:, 1]
sel = p_te >= corte_val
lucro_te = y_te.values[sel].sum() * SUCESSO * VALOR - sel.sum() * CUSTO
print(f"Corte escolhido na validação: {corte_val:.2f} -> lucro no teste R$ {lucro_te:,.0f}")

Corte escolhido na validação: 0.39 -> lucro no teste R$ 3,474


Desafio 6: feature engineering

Última etapa da lista. Crie uma variável nova, horas_por_dia_ativo, e veja se ela ajuda. Este código compara os dois modelos na mesma divisão e funciona sozinho, desde que X_tr, X_te, y_tr e y_te existam:

In [42]:
from sklearn.metrics import average_precision_score

def pr_auc(Xa, Xb, nome):
    cat_ = Xa.select_dtypes("object").columns.tolist()
    num_ = [c for c in Xa.columns if c not in cat_]
    pre_ = ColumnTransformer([
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="ND")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_)])
    m = Pipeline([("pre", pre_), ("m", LogisticRegression(C=0.5, max_iter=2000))]).fit(Xa, y_tr)
    s = average_precision_score(y_te, m.predict_proba(Xb)[:, 1])
    print(f"{nome}: PR-AUC {s:.4f}")
    return s

def cria_features(d):
    d = d.copy()
    d["horas_por_dia_ativo"] = np.where(d["dias_ativos_30d"] > 0,
                                        d["horas_assistidas_30d"] / d["dias_ativos_30d"], 0)
    return d

base = pr_auc(X_tr, X_te, "Sem a feature nova")
nova = pr_auc(cria_features(X_tr), cria_features(X_te), "Com horas_por_dia_ativo")
print(f"Diferença: {nova - base:+.4f}")

Sem a feature nova: PR-AUC 0.4049
Com horas_por_dia_ativo: PR-AUC 0.4049
Diferença: -0.0000
